# MCQ Generator

Generates multiple choice questions (MCQs) from a piece of text using Google Gemini via LangChain, then asks the model to review the quiz.

The flow is:
1. **Generate** – send the text to the LLM and get back a quiz as JSON.
2. **Evaluate** – send that quiz back to the LLM for a short complexity review and suggested improvements.
3. **Pipeline** – chain both steps so one call runs the whole thing.

## 1. Imports

In [1]:
# Standard library
import os
import json
import traceback

# Data handling
import pandas as pd

# Loads API keys from the .env file
from dotenv import load_dotenv

# Google Gemini chat model
from langchain_google_genai import ChatGoogleGenerativeAI

# Prompt templates
from langchain_core.prompts import PromptTemplate

# Output parsers: plain text and JSON
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

# Building blocks for chaining steps together
from langchain_core.runnables import (
    RunnableSequence,
    RunnablePassthrough,
    RunnableLambda,
)

# PDF processing
from pypdf import PdfReader

# Callback handler for usage metadata
from langchain_core.callbacks import UsageMetadataCallbackHandler
cb = UsageMetadataCallbackHandler()

## 2. Environment setup

Reads the API keys from `.env`. The printout only shows whether each key was found, never the key itself.

In [2]:
load_dotenv()

KEY_OPENAI = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# True means the key was loaded
print(f"OPENAI_API_KEY: {bool(KEY_OPENAI)}")
print(f"GEMINI_API_KEY: {bool(GEMINI_API_KEY)}")

OPENAI_API_KEY: True
GEMINI_API_KEY: True


## 3. LLM setup

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.2,          # low temperature = more consistent, less random output
    max_output_tokens=8192,   # cap on response length; 2048 cut off the rewritten quiz mid-JSON
    api_key=GEMINI_API_KEY,
)

## 4. Step 1: Generate the quiz

`RESPONSE_JSON` is an example of the shape we want back. It's passed into the prompt so the model copies that structure, one entry per question.

In [4]:
# Expected output format for each question
RESPONSE_JSON = {
    "1": {
        'mcq': 'multiple choice question',
        'options': {
            'a': 'choice here',
            'b': 'choice here',
            'c': 'choice here',
            'd': 'choice here'
        },
        'correct': 'correct answer',
    },
}

In [5]:
# Prompt for generating the quiz. Values in {braces} are filled in at run time.
TEMPLATE = """
Text: {text}
You are an expert MCQ generator. Given the above text, it's your job to
generate {number} multiple choice questions for the subject {subject}. The questions should be in a {tone} tone.
Make sure to provide the output in the following RESPONSE_JSON format: {response_json}
"""

quiz_prompt_template = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [6]:
# prompt -> LLM -> parse the reply into a Python dict
quiz_chain = (
    quiz_prompt_template
    | llm
    | JsonOutputParser()
)

## 5. Step 2: Evaluate the quiz

Sends the generated quiz back to the LLM for a short complexity review (max 50 words) and asks it to rewrite any questions that don't fit the chosen tone. The reply comes back as JSON with two keys: `complexity_analysis` and `updated_quiz`.

In [7]:
# Prompt for reviewing the quiz. Literal JSON braces are doubled ({{ }}) so PromptTemplate doesn't treat them as variables.
TEMPLATE2 = """
You are an expert MCQ generator. Given the Multiple Choice Quiz for the subject {subject}, you need to evaluate the complexity of the questions and give a complete analysis of the quiz.
Only use at MAX 50 words for complexity analysis.
If the quiz is not complex, update the questions that need to be improved to the tone {tone} so that it perfectly fits the students.
Quiz: {quiz}

Return ONLY valid JSON (no markdown, no code fences) in this format:
{{
    "complexity_analysis": "your analysis here",
    "updated_quiz": <the full quiz in the same format as the input quiz, with any improved questions swapped in>
}}
"""

quiz_evaluation_prompt_template = PromptTemplate(
    input_variables=["quiz", "subject", "tone"],
    template=TEMPLATE2
)

In [8]:
# prompt -> LLM -> parse the reply into a Python dict
quiz_evaluation_chain = (
    quiz_evaluation_prompt_template
    | llm
    | JsonOutputParser()
)

## 6. Full pipeline

Runs generation, then evaluation, in one call. The result is a dict holding the original inputs plus `quiz` and `evaluation`.

In [9]:
# Each RunnablePassthrough.assign keeps everything received so far and adds one new key:
sequential_chain = (
    RunnablePassthrough.assign(
        quiz=quiz_chain                  # 1. generate the quiz -> "quiz"
    )
    | RunnablePassthrough.assign(
        evaluation=quiz_evaluation_chain # 2. review it (uses quiz, subject, tone) -> "evaluation"
    )
)

In [10]:
file_path = '../data.txt'

In [11]:
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

In [12]:
# json.dump{
#     RESPONSE_JSON
# }

# print("RESPONSE_JSON:", RESPONSE_JSON)

## 7. Run it

In [ ]:
response = sequential_chain.invoke({
    "text": "Java is a high-level, class-based, object-oriented programming language that is designed to have as few implementation dependencies as possible. It is a general-purpose programming language intended to let application developers write once, run anywhere (WORA), meaning that compiled Java code can run on all platforms that support Java without the need for recompilation.",
    "number": 5,
    "subject": "Java Programming",
    "tone": "Intermediate",
    "response_json": json.dumps(RESPONSE_JSON),  # the quiz prompt needs this to know the output format
})

quiz = response["quiz"]               # dict of questions
evaluation = response["evaluation"]   # review text

In [ ]:
print("GENERATED QUIZ:")
print(response["quiz"])

print("\nEVALUATION:")
print(response["evaluation"])

print("\nTOKEN USAGE:")
# print(cb.usage_metadata)

GENERATED QUIZ:
{'1': {'mcq': "What does the concept of 'Write Once, Run Anywhere' (WORA) specifically mean in the context of Java development?", 'options': {'a': 'Java source code automatically translates into machine code for all operating systems before execution.', 'b': 'Compiled Java code can execute on any platform supporting Java without requiring recompilation.', 'c': 'Java applications must be compiled separately for each target operating system.', 'd': 'Java programs run directly on hardware without needing a platform-specific environment.'}, 'correct': 'b'}, '2': {'mcq': 'According to the text, Java was designed to minimize which of the following?', 'options': {'a': 'High-level object abstractions', 'b': 'Implementation dependencies', 'c': 'Cross-platform compatibility', 'd': 'Class-based structures'}, 'correct': 'b'}, '3': {'mcq': 'How is Java classified regarding its language features and paradigm?', 'options': {'a': 'Low-level, procedural, and platform-dependent', 'b': 'H

In [ ]:
# evaluation is already a dict (JsonOutputParser did the json.loads for us)
print(evaluation["complexity_analysis"])

updated_quiz = evaluation["updated_quiz"]
print(json.dumps(updated_quiz, indent=2))

TypeError: string indices must be integers, not 'str'